# 02 — Generative Adversarial Networks: Vanilla GAN (MLP) vs DCGAN

Objektif minimax (Goodfellow et al., 2014):
$$\min_G\max_D\ \mathbb{E}_{x\sim p_{data}}[\log D(x)] + \mathbb{E}_{z\sim\mathcal{N}(0,I)}[\log(1-D(G(z)))]$$

* **Discriminator**: $\mathcal{L}_D = -\mathbb{E}[\log D(x)] - \mathbb{E}[\log(1-D(G(z)))]$ (BCE)
* **Generator** (non-saturating): $\mathcal{L}_G = -\mathbb{E}[\log D(G(z))]$

Kedua GAN memakai pengaturan yang sama: $z\in\mathbb{R}^{100}$, Adam (lr = 2e-4, β = (0.5, 0.999)), batch 128, 30 epoch,
dan citra diskalakan ke [-1, 1]. Perbedaannya hanya pada **arsitektur**: fully connected vs. konvolusional dengan
BatchNorm, strided conv, dan inisialisasi N(0, 0.02).

In [ ]:
# ===== Import library =====
import torch
import torch.nn as nn                 # modul neural network (loss BCE, dll.)
import matplotlib.pyplot as plt
from collections import defaultdict   # dict otomatis untuk mencatat riwayat loss

# Konstanta & helper bersama (SEED, DEVICE, get_loaders, to_tanh_range, dll.)
from common import *
# Arsitektur GAN: versi MLP (Vanilla GAN) dan versi konvolusi (DCGAN)
from models import (MLPGenerator, MLPDiscriminator, DCGANGenerator, DCGANDiscriminator,
                    dcgan_weights_init)

set_seed(SEED)                                                   # kunci keacakan
train_loader, val_loader, test_loader = get_loaders(BATCH_SIZE)  # data 80/10/10

# ===== Hyperparameter =====
EPOCHS = 1 if QUICK else 30       # 30 epoch (1 jika mode uji cepat)
Z_DIM = 100                       # ukuran vektor noise z input generator
LR, BETAS = 2e-4, (0.5, 0.999)    # setting Adam standar untuk GAN (beta1=0.5 agar lebih stabil)

g = torch.Generator().manual_seed(SEED)   # generator acak khusus agar z tetap selalu sama
FIXED_Z = torch.randn(64, Z_DIM, generator=g).to(DEVICE)   # untuk memantau evolusi sampel
PROBE_Z = torch.randn(256, Z_DIM, generator=g).to(DEVICE)  # untuk probe gradien

## Training loop GAN
Selain loss D dan G per iterasi, dicatat juga:
* $\bar D(x)$ dan $\bar D(G(z))$ per epoch, untuk melihat dominasi discriminator;
* **probe vanishing gradient**: norma gradien $\lVert\nabla_{\theta_G}\rVert$ untuk loss *saturating* $\log(1-D(G(z)))$
  dibandingkan loss *non-saturating* $-\log D(G(z))$.

In [ ]:
def grad_norm(params):
    # Norma L2 seluruh gradien: sqrt(jumlah kuadrat semua gradien) -> ukuran "seberapa besar" gradien
    return torch.sqrt(sum((p.grad.detach() ** 2).sum() for p in params if p.grad is not None)).item()


def gradient_probe(G, D):
    """Norma gradien G untuk dua formulasi loss generator pada batch z tetap."""
    # Tujuan: membuktikan masalah vanishing gradient pada loss "saturating"
    out = {}
    for kind in ("saturating", "non_saturating"):
        G.zero_grad(set_to_none=True)
        logits = D(G(PROBE_Z))            # skor discriminator untuk gambar palsu
        if kind == "saturating":      # minimize log(1 - D(G(z)))
            # Loss asli dari paper; gradien hampir nol saat D yakin gambar palsu (D(G(z)) ~ 0)
            loss = torch.log1p(-torch.sigmoid(logits).clamp(max=1 - 1e-7)).mean()
        else:                         # minimize -log D(G(z))
            # Versi non-saturating yang dipakai di training (gradien tetap besar)
            loss = nn.functional.binary_cross_entropy_with_logits(logits, torch.ones_like(logits))
        loss.backward()
        out[kind] = grad_norm(G.parameters())   # catat besar gradien generator
    # Bersihkan gradien agar tidak mengganggu training
    G.zero_grad(set_to_none=True); D.zero_grad(set_to_none=True)
    return out


def train_gan(G, D, name, epochs=EPOCHS):
    set_seed(SEED)
    G, D = G.to(DEVICE), D.to(DEVICE)
    # Dua optimizer terpisah: satu untuk Generator, satu untuk Discriminator
    optG = torch.optim.Adam(G.parameters(), lr=LR, betas=BETAS)
    optD = torch.optim.Adam(D.parameters(), lr=LR, betas=BETAS)
    bce = nn.BCEWithLogitsLoss()   # BCE + sigmoid sekaligus (lebih stabil secara numerik)
    hist = defaultdict(list)       # riwayat loss & statistik
    snapshots = {}                 # sampel gambar di epoch tertentu
    for ep in range(1, epochs + 1):
        G.train(); D.train()
        sums, n = defaultdict(float), 0
        with EpochTimer() as tm:
            for x, _ in train_loader:
                x = to_tanh_range(x.to(DEVICE))   # piksel [0,1] -> [-1,1] (sesuai output Tanh generator)
                b = x.size(0)
                ones = torch.ones(b, device=DEVICE)    # label "asli" = 1
                zeros = torch.zeros(b, device=DEVICE)  # label "palsu" = 0
                fake = G(torch.randn(b, Z_DIM, device=DEVICE))   # generator membuat gambar palsu dari noise

                # --- update Discriminator ---
                # D belajar: gambar asli -> 1, gambar palsu -> 0
                # fake.detach(): gradien tidak mengalir ke G saat melatih D
                d_real, d_fake = D(x), D(fake.detach())
                loss_d = bce(d_real, ones) + bce(d_fake, zeros)
                optD.zero_grad(set_to_none=True)
                loss_d.backward()
                optD.step()

                # --- update Generator (non-saturating) ---
                # G belajar menipu D: ingin D menilai gambar palsu sebagai "asli" (label 1)
                loss_g = bce(D(fake), ones)
                optG.zero_grad(set_to_none=True)
                loss_g.backward()
                gn = torch.sqrt(sum((p.grad ** 2).sum() for p in G.parameters()))   # besar gradien G
                optG.step()

                # Catat loss per iterasi & akumulasi untuk rata-rata epoch
                ld, lg = loss_d.detach(), loss_g.detach()
                hist["iter_d_loss"].append(ld); hist["iter_g_loss"].append(lg)
                sums["d_loss"] += ld; sums["g_loss"] += lg; sums["g_grad_norm"] += gn.detach()
                # Rata-rata probabilitas D: D(x) untuk gambar asli, D(G(z)) untuk gambar palsu
                sums["d_real"] += torch.sigmoid(d_real).mean().detach()
                sums["d_fake"] += torch.sigmoid(d_fake).mean().detach()
                n += 1
        # Simpan rata-rata per epoch
        hist["epoch_time"].append(tm.seconds)
        for k, v in sums.items():
            hist[k].append(float(v) / n)
        # Probe vanishing gradient di akhir tiap epoch
        probe = gradient_probe(G, D)
        hist["probe_grad_saturating"].append(probe["saturating"])
        hist["probe_grad_non_saturating"].append(probe["non_saturating"])
        # Simpan 16 sampel dari z tetap pada epoch tertentu -> melihat perkembangan kualitas gambar
        if ep in (1, 5, 10, 20, epochs):
            G.eval()
            with torch.no_grad():
                snapshots[ep] = from_tanh_range(G(FIXED_Z[:16])).cpu()
        # Cetak progres
        if ep == 1 or ep % 5 == 0 or ep == epochs:
            print(f"[{name}] epoch {ep:02d} | D {hist['d_loss'][-1]:.3f} | G {hist['g_loss'][-1]:.3f} "
                  f"| D(x) {hist['d_real'][-1]:.2f} D(G(z)) {hist['d_fake'][-1]:.2f} "
                  f"| grad sat/non-sat {probe['saturating']:.2e}/{probe['non_saturating']:.2e} "
                  f"| {tm.seconds:.1f}s")
    # Ubah tensor menjadi float biasa agar bisa disimpan ke JSON
    hist["iter_d_loss"] = [float(v) for v in hist["iter_d_loss"]]
    hist["iter_g_loss"] = [float(v) for v in hist["iter_g_loss"]]
    # Simpan bobot G & D, snapshot, dan riwayat training
    torch.save(G.state_dict(), os.path.join(CKPT_DIR, f"{name}_G.pt"))
    torch.save(D.state_dict(), os.path.join(CKPT_DIR, f"{name}_D.pt"))
    torch.save(snapshots, os.path.join(RESULTS_DIR, f"snapshots_{name}.pt"))
    save_json(dict(hist), f"history_{name}.json")
    return G, D, hist, snapshots

## Vanilla GAN (fully connected)

In [ ]:
# Latih Vanilla GAN (generator & discriminator berupa MLP / fully connected)
G_mlp, D_mlp, hist_gan, snap_gan = train_gan(MLPGenerator(Z_DIM), MLPDiscriminator(), "vanilla_gan")
# Tampilkan jumlah parameter generator dan discriminator
print("params G/D:", count_params(G_mlp), count_params(D_mlp))

## DCGAN

In [ ]:
# Buat generator & discriminator DCGAN (berbasis konvolusi)
G_dc, D_dc = DCGANGenerator(Z_DIM), DCGANDiscriminator()
set_seed(SEED)
# Inisialisasi bobot N(0, 0.02) sesuai paper DCGAN
G_dc.apply(dcgan_weights_init); D_dc.apply(dcgan_weights_init)
# Latih dengan training loop yang sama persis dengan Vanilla GAN
G_dc, D_dc, hist_dc, snap_dc = train_gan(G_dc, D_dc, "dcgan")
print("params G/D:", count_params(G_dc), count_params(D_dc))

## Evolusi sampel selama training (z tetap)

In [ ]:
# Daftar epoch yang punya snapshot (1, 5, 10, 20, 30)
eps_ = sorted(snap_gan)
# Grid 2 baris (Vanilla GAN, DCGAN) x jumlah snapshot
fig, axes = plt.subplots(2, len(eps_), figsize=(3 * len(eps_), 6.4))
axes = axes.reshape(2, -1)
for j, ep in enumerate(eps_):
    # Karena z tetap (FIXED_Z), perubahan gambar murni menunjukkan perkembangan generator
    show_grid(snap_gan[ep], nrow=4, ax=axes[0, j], title=f"Vanilla GAN — epoch {ep}")
    show_grid(snap_dc[ep], nrow=4, ax=axes[1, j], title=f"DCGAN — epoch {ep}")
plt.tight_layout()
# Simpan figur ke folder figures/
plt.savefig(os.path.join(FIG_DIR, "figS3_gan_sample_evolution.png"), dpi=200, bbox_inches="tight")
plt.show()